In [23]:
import torch
from einops import rearrange, einsum

# ===================== 场景 1：矩阵乘法 =====================
# 1.1 普通二维矩阵乘法 (2, 2) @ (2, 2)
a = torch.tensor([[1, 2],
                  [3, 4]])
b = torch.tensor([[1, 1],
                  [1, 1]])

out_mat = a @ b
print(f"a @ b 矩阵乘法结果:\n{out_mat}, 形状：{out_mat.shape}\n")

a @ b 矩阵乘法结果:
tensor([[3, 3],
        [7, 7]]), 形状：torch.Size([2, 2])



In [ ]:
# 1.2 带 batch 的矩阵乘法广播 (batch=2, 2, 2) @ (2, 2)
batch_a = torch.tensor([
    [[1, 2],
     [3, 4]],
    [[-1, -2],
     [-1, -4]]
])  # (2, 2, 2)

w = torch.tensor([[1, 1],
                  [0, 1]])  # (2, 2)

out_batch = batch_a @ w # 矩阵乘法会被自动广播到前面的所有维度上，batch 中每一个 (2, 2) 矩阵独立地和 w 做矩阵乘法
print(f"batch_a @ w 矩阵乘法结果:\n{out_batch}, 形状：{out_batch.shape}")

# 那如果是 A: (3, 2, 2, 2) 和 B: (2, 2) 做矩阵乘法运算呢？(3, 2, 2, 2)
# 广播机制，无非就是对前面所有的批次维度上的元素做相同的操作

# einsum 优雅写法：输入输出维度映射清清楚楚，自动处理任意前置 batch 维度的广播
out_ein = einsum(batch_a, w, "b n k,  k m -> b n m")
print(f"\neinsum 矩阵乘法结果:\n{out_ein}, 形状：{out_ein.shape}\n")

batch_a @ w 矩阵乘法结果:
tensor([[[ 1,  3],
         [ 3,  7]],

        [[-1, -3],
         [-1, -5]]]), 形状：torch.Size([2, 2, 2])

einsum 矩阵乘法结果:
tensor([[[ 1,  3],
         [ 3,  7]],

        [[-1, -3],
         [-1, -5]]]), 形状：torch.Size([2, 2, 2])



In [25]:
# ===================== 场景 2：逐点乘法 =====================
m = torch.tensor([[1, 4],
                  [6, 7]])  # (2, 2)
v = torch.tensor([[2, 2],
                  [2, 2]]) # （2, 2)
out_elem = m * v
print(f"m * v 逐点乘结果:\n{out_elem}, 形状：{out_elem.shape}\n")

v1 = torch.tensor([2, 2]) # （2)
# 逐点乘法广播：v1 会被复制扩展为(2, 2)，再对应位置元素相乘
print(f"m * v1 广播逐点乘结果:\n{out_elem}, 形状：{out_elem.shape}\n")

# 广播遵循从右往左对齐，从第一个不对齐的位置开始扩展
# (4, 3, 3, 2) a
#       (3, 2) b
#    (3, 3, 2) b1
# (4, 3, 3, 2) b2

m * v 逐点乘结果:
tensor([[ 2,  8],
        [12, 14]]), 形状：torch.Size([2, 2])

m * v1 广播逐点乘结果:
tensor([[ 2,  8],
        [12, 14]]), 形状：torch.Size([2, 2])



In [26]:
# ===================== 场景 3：归约操作之求和，沿最后一维 dim=-1 求和 =====================
t = torch.tensor([[1, 2],
                  [3, 4],
                  [5, 6]])
print(f"t:\n{t}, 形状：{t.shape}")
sum_last_keepdim = t.sum(dim=-1, keepdim=True)
# dim=-1 代表沿着最后一个维度做求和，该维度会被消除；归约简单来说就是将 n 个元素按照某种操作合并成 1 个元素；
print(f"\n沿最后一维求和且保留最后一个维度:\n{sum_last_keepdim}, 形状：{sum_last_keepdim.shape}\n")

sum_last = t.sum(dim=-1, keepdim=False) # 默认 keepdim=False，不保留归约后的维度
print(f"沿最后一维求和且不保留归约维度:\n{sum_last}, 形状：{sum_last.shape}\n")

t:
tensor([[1, 2],
        [3, 4],
        [5, 6]]), 形状：torch.Size([3, 2])

沿最后一维求和且保留最后一个维度:
tensor([[ 3],
        [ 7],
        [11]]), 形状：torch.Size([3, 1])

沿最后一维求和且不保留归约维度:
tensor([ 3,  7, 11]), 形状：torch.Size([3])



In [ ]:
# ===================== 场景 4：维度拆分与重排 =====================
# (batch, seq, d_model), d_model=8 拆分为 h=2, d=4
x = torch.tensor([
    [[1, 2, 3, 4, 5, 6, 7, 8],
     [9,10,11,12,13,14,15,16]]
])  # shape (batch=1, seq=2, d_model=8)
print(f"x:\n{x}, 形状：{x.shape}")

# 传统抽象写法：view做维度拆分，再transpose交换维度，难读且易错
x_trad = x.view(1, 2, 2, 4).transpose(1, 2)
print(f"view+transpose 传统写法结果:\n{x_trad}, 形状：{x_trad.shape}")

# rearrange优雅写法：显式声明维度拆分与顺序变换，替代view+transpose
x_rearr = rearrange(x, "b s (h d) -> b h s d", h=2, d=4)
print(f"rearrange优雅写法结果:\n{x_rearr}, 形状：{x_rearr.shape}")